# Getting started: Australian land parcels + G-NAF

Every land parcel from the open state cadastres (NSW, VIC, QLD, TAS, ACT, WA), one file per state, each parcel
carrying the ids of its G-NAF addresses. This notebook loads the data, compares states, plots parcel polygons and
joins addresses.

The address half of the notebook uses the companion dataset
[Australian Addresses (G-NAF)](https://www.kaggle.com/datasets/museofnephorion/australian-addresses-gnaf): the same G-NAF release with every address flattened to one row. Both
are inputs of this notebook; if you copy it without the G-NAF input, the address cells say so and skip.

The datasets are built by the open-source pipeline at
[github.com/nephorion/kaggle-au-cadastre](https://github.com/nephorion/kaggle-au-cadastre).

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

# Locate the datasets wherever Kaggle mounts them.
DATA = os.path.dirname(next(iter(glob.glob("/kaggle/input/**/parquet", recursive=True))))
GNAF = next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/gnaf_full.parquet", recursive=True)),
            os.environ.get("GNAF_DATA"))   # the companion G-NAF dataset, an optional second input
print("parcels:", DATA)
print("G-NAF:", GNAF)
sorted(os.listdir(f"{DATA}/parquet"))

## Load one state without geometry

Parquet reads only the columns you ask for. Leave out `wkt` unless you need the polygons.

In [ ]:
cols = ["state", "source_oid", "gnaf_legal_parcel_id", "gnaf_key_matched", "gnaf_address_count",
        "gnaf_address_detail_pids", "lot", "plan", "tenure", "area_m2", "locality", "point_lon", "point_lat"]
act = pd.read_parquet(f"{DATA}/parquet/ACT_parcels.parquet", columns=cols)
print(f"{len(act):,} ACT parcels")
act.head()

## All states in one table

The identity, G-NAF and harmonised columns are the same in every file, so states stack directly.

In [ ]:
common = ["state", "gnaf_key_matched", "gnaf_address_count", "status", "tenure", "is_strata", "area_m2"]
files = sorted(glob.glob(f"{DATA}/parquet/*_parcels.parquet"))
australia = pd.concat([pd.read_parquet(f, columns=common) for f in files], ignore_index=True)

by_state = australia.groupby("state").agg(
    parcels=("state", "size"),
    with_address=("gnaf_address_count", lambda s: (s > 0).mean()),
    key_matched=("gnaf_key_matched", "mean"),
    median_area_m2=("area_m2", "median"),
)
by_state.style.format({"parcels": "{:,}", "with_address": "{:.1%}", "key_matched": "{:.1%}",
                       "median_area_m2": "{:,.0f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
(australia["gnaf_address_count"].clip(upper=10).value_counts().sort_index()
    .plot.bar(ax=ax, color="#2a9d8f", title="Addresses per parcel, all states (10 = 10 or more)"))
ax.set_xlabel("G-NAF addresses on the parcel")
ax.set_ylabel("parcels")
plt.show()

## Plot parcel polygons

`wkt` holds each parcel's boundary in EPSG:4326. Here are the parcels of one Canberra suburb, coloured by the
number of addresses on each.

In [ ]:
import geopandas as gpd

act_geo = pd.read_parquet(f"{DATA}/parquet/ACT_parcels.parquet",
                          columns=["source_oid", "locality", "gnaf_address_count", "wkt"])
suburb = act_geo[act_geo["locality"] == "BRADDON"]
gdf = gpd.GeoDataFrame(suburb, geometry=gpd.GeoSeries.from_wkt(suburb["wkt"]), crs="EPSG:4326")

ax = gdf.plot(column=gdf["gnaf_address_count"].clip(upper=20), cmap="viridis", legend=True,
              linewidth=0.2, edgecolor="white", figsize=(8, 8))
ax.set_title(f"Braddon, ACT: {len(gdf):,} parcels by address count")
ax.set_axis_off()
plt.show()

## Join G-NAF addresses

`gnaf_address_detail_pids` is a list per parcel: explode it to one row per address and merge with
[Australian Addresses (G-NAF)](https://www.kaggle.com/datasets/museofnephorion/australian-addresses-gnaf). Its `gnaf_csv/` folder splits the addresses by state and postcode
district (`gnaf_ACT_26xx.csv`, ...), so reading one state stays light. Every column is text, so ids and postcodes
keep their leading zeros.

In [ ]:
if GNAF:
    gnaf_cols = ["address_detail_pid", "full_address", "locality_name", "postcode_effective",
                 "longitude", "latitude", "geocode_type_name"]
    gnaf = pd.concat([pd.read_csv(f, usecols=gnaf_cols, dtype=str)
                      for f in sorted(glob.glob(f"{GNAF}/gnaf_csv/gnaf_ACT_*.csv"))], ignore_index=True)
    gnaf[["longitude", "latitude"]] = gnaf[["longitude", "latitude"]].astype(float)
    print(f"{len(gnaf):,} ACT addresses")

    parcel_addresses = (act.explode("gnaf_address_detail_pids")
                           .merge(gnaf, how="left", left_on="gnaf_address_detail_pids",
                                  right_on="address_detail_pid"))
    display(parcel_addresses[["source_oid", "gnaf_legal_parcel_id", "full_address", "locality_name"]]
            .dropna(subset=["full_address"]).head(10))
else:
    print("Add the Australian Addresses (G-NAF) dataset as a second input to run the address cells.")

## From an address to its parcel

The same link read the other way: every address gets the parcel it sits on. Addresses with no parcel are mostly
on land the open cadastre leaves out, or in strata buildings whose lots carry no plan key.

In [ ]:
if GNAF:
    pid_to_parcel = (act[["source_oid", "gnaf_address_detail_pids", "tenure", "area_m2"]]
                     .explode("gnaf_address_detail_pids").dropna(subset=["gnaf_address_detail_pids"])
                     .rename(columns={"gnaf_address_detail_pids": "address_detail_pid"}))
    address_parcels = gnaf.merge(pid_to_parcel, how="left", on="address_detail_pid")
    print(f"{address_parcels['source_oid'].notna().mean():.1%} of ACT addresses are on a parcel")
    display(address_parcels.groupby("locality_name")["source_oid"]
            .agg(addresses="size", on_a_parcel=lambda s: s.notna().mean())
            .sort_values("addresses", ascending=False).head(10)
            .style.format({"addresses": "{:,}", "on_a_parcel": "{:.1%}"}))

## Addresses on the map

G-NAF's geocodes over Braddon's parcels. Each address point is coloured by whether it is linked to a parcel.

In [ ]:
if GNAF:
    pts = address_parcels[address_parcels["locality_name"] == "BRADDON"]
    linked = pts["source_oid"].notna()
    ax = gdf.plot(color="#e9ecef", linewidth=0.3, edgecolor="#adb5bd", figsize=(8, 8))
    ax.scatter(pts.loc[linked, "longitude"], pts.loc[linked, "latitude"], s=4, color="#2a9d8f",
               label=f"linked to a parcel ({linked.sum():,})")
    ax.scatter(pts.loc[~linked, "longitude"], pts.loc[~linked, "latitude"], s=4, color="#e76f51",
               label=f"no parcel ({(~linked).sum():,})")
    ax.set_title("Braddon, ACT: G-NAF addresses over parcels")
    ax.legend(loc="lower left", frameon=False)
    ax.set_axis_off()
    plt.show()